# 01 - Data Understanding and Cleaning

This notebook inspects the raw NASA FIRMS multi-sensor wildfire detections dataset, standardizes
the schema, validates coordinates and timestamps, normalizes confidence values, and writes the
cleaned analytical dataset to `data/processed/cleaned_wildfire_data.csv`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from data_utils import (
    build_cleaned_dataset,
    find_raw_dataset,
    read_dataset,
    standardize_columns,
)

## Locate and preview the raw dataset

In [2]:
raw_path = find_raw_dataset(ROOT / "data" / "raw")
raw_path

WindowsPath('d:/Codingan Pribadi/SERIUS/NASA FIRM/wildfire-risk-intelligence/data/raw/nasa_firms_multisensor_2026.csv')

In [3]:
raw_df = read_dataset(raw_path)
print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]:,}")
display(raw_df.head())

Rows: 565,708
Columns: 22


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,frp,daynight,source_dataset,brightness,bright_t31,year,month,day,season,lat_band
0,59.13006,37.79127,315.65,0.51,0.50,2026-04-21,50,N21,VIIRS,n,...,2.91,Night,VIIRS_NOAA21_NRT,NaN,NaN,2026,4,21,Spring,30°N-60°N
1,59.14488,37.84920,295.03,0.52,0.50,2026-04-21,50,N21,VIIRS,n,...,2.58,Night,VIIRS_NOAA21_NRT,NaN,NaN,2026,4,21,Spring,30°N-60°N
2,59.15345,37.83788,317.23,0.52,0.50,2026-04-21,50,N21,VIIRS,n,...,2.73,Night,VIIRS_NOAA21_NRT,NaN,NaN,2026,4,21,Spring,30°N-60°N
3,59.96397,45.70111,300.81,0.41,0.60,2026-04-21,50,N21,VIIRS,n,...,0.58,Night,VIIRS_NOAA21_NRT,NaN,NaN,2026,4,21,Spring,30°N-60°N
4,61.14549,28.79877,297.11,0.41,0.37,2026-04-21,50,N21,VIIRS,n,...,1.10,Night,VIIRS_NOAA21_NRT,NaN,NaN,2026,4,21,Spring,60°N-90°N


## Schema, missing values, duplicates, and statistics

In [4]:
profile_table = pd.DataFrame({
    "column": raw_df.columns,
    "dtype": raw_df.dtypes.astype(str).values,
    "missing_values": raw_df.isna().sum().values,
    "missing_pct": (raw_df.isna().mean().values * 100).round(2),
    "unique_values": [raw_df[col].nunique(dropna=True) for col in raw_df.columns],
})
display(profile_table)
print(f"Duplicate rows: {raw_df.duplicated().sum():,}")
display(raw_df.describe(include="all").T.head(30))

,column,dtype,missing_values,missing_pct,unique_values
0,latitude,float64,0,0.00,527144
1,longitude,float64,0,0.00,536920
2,bright_ti4,float64,30665,5.42,6459
3,scan,float64,0,0.00,317
4,track,float64,0,0.00,145
5,acq_date,object,0,0.00,5
6,acq_time,int64,0,0.00,706
7,satellite,object,0,0.00,5
8,instrument,object,0,0.00,2
9,confidence,object,0,0.00,104


Duplicate rows: 18


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
latitude,565708.0,NaN,NaN,NaN,15.735117,21.103543,-86.59821,8.77737,19.712815,26.895128,72.05147
longitude,565708.0,NaN,NaN,NaN,52.543841,69.724298,-175.07068,9.349215,79.28141,99.461658,176.51857
bright_ti4,535043.0,NaN,NaN,NaN,333.357302,17.598792,207.38,326.22,336.36,344.19,367.0
scan,565708.0,NaN,NaN,NaN,0.513917,0.288916,0.3,0.4,0.44,0.52,4.82
track,565708.0,NaN,NaN,NaN,0.519432,0.197378,0.36,0.38,0.46,0.59,2.0
acq_date,565708,5,2026-04-22,128907,NaN,NaN,NaN,NaN,NaN,NaN,NaN
acq_time,565708.0,NaN,NaN,NaN,1128.647739,594.794423,1.0,652.0,913.0,1737.0,2358.0
satellite,565708,5,N,181989,NaN,NaN,NaN,NaN,NaN,NaN,NaN
instrument,565708,2,VIIRS,535043,NaN,NaN,NaN,NaN,NaN,NaN,NaN
confidence,565708,104,n,435757,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Standardize columns and run the cleaning pipeline

In [5]:
standardized_preview = standardize_columns(raw_df.head())
standardized_preview.columns.tolist()

['latitude',
 'longitude',
 'bright_ti4',
 'scan',
 'track',
 'acq_date',
 'acq_time',
 'satellite',
 'instrument',
 'confidence',
 'version',
 'bright_ti5',
 'frp',
 'daynight',
 'source_dataset',
 'brightness',
 'bright_t31',
 'year',
 'month',
 'day',
 'season',
 'lat_band']

In [6]:
clean_df, cleaning_profile, clean_path = build_cleaned_dataset(
    raw_path=raw_path,
    output_path=ROOT / "data" / "processed" / "cleaned_wildfire_data.csv",
)
cleaning_profile, clean_path

({'raw_rows': 565708,
  'raw_columns': 22,
  'rows_after_coordinate_datetime_validation': 565690,
  'duplicate_rows_removed': 18,
  'final_rows': 565690,
  'final_columns': 28},
 WindowsPath('d:/Codingan Pribadi/SERIUS/NASA FIRM/wildfire-risk-intelligence/data/processed/cleaned_wildfire_data.csv'))

## Validate cleaned data

In [7]:
print(f"Clean rows: {len(clean_df):,}")
print(f"Date range: {clean_df['acq_datetime'].min()} to {clean_df['acq_datetime'].max()}")
print(f"Latitude range: {clean_df['latitude'].min():.3f} to {clean_df['latitude'].max():.3f}")
print(f"Longitude range: {clean_df['longitude'].min():.3f} to {clean_df['longitude'].max():.3f}")
display(clean_df.head())

Clean rows: 565,690
Date range: 2026-04-21 00:01:00+00:00 to 2026-04-25 20:28:00+00:00
Latitude range: -86.598 to 72.051
Longitude range: -175.071 to 176.519


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,month,day,season,lat_band,acq_datetime,acq_date_clean,confidence_score,confidence_label,brightness_primary,week
0,59.13006,37.79127,315.65,0.51,0.50,2026-04-21,50,N21,VIIRS,n,...,4,21,Spring,30°N-60°N,2026-04-21 00:50:00+00:00,2026-04-21,66.0,Nominal,315.65,17
1,59.14488,37.84920,295.03,0.52,0.50,2026-04-21,50,N21,VIIRS,n,...,4,21,Spring,30°N-60°N,2026-04-21 00:50:00+00:00,2026-04-21,66.0,Nominal,295.03,17
2,59.15345,37.83788,317.23,0.52,0.50,2026-04-21,50,N21,VIIRS,n,...,4,21,Spring,30°N-60°N,2026-04-21 00:50:00+00:00,2026-04-21,66.0,Nominal,317.23,17
3,59.96397,45.70111,300.81,0.41,0.60,2026-04-21,50,N21,VIIRS,n,...,4,21,Spring,30°N-60°N,2026-04-21 00:50:00+00:00,2026-04-21,66.0,Nominal,300.81,17
4,61.14549,28.79877,297.11,0.41,0.37,2026-04-21,50,N21,VIIRS,n,...,4,21,Spring,60°N-90°N,2026-04-21 00:50:00+00:00,2026-04-21,66.0,Nominal,297.11,17


In [8]:
validation_summary = {
    "invalid_latitude": (~clean_df["latitude"].between(-90, 90)).sum(),
    "invalid_longitude": (~clean_df["longitude"].between(-180, 180)).sum(),
    "missing_datetime": clean_df["acq_datetime"].isna().sum(),
    "duplicate_rows": clean_df.duplicated().sum(),
}
validation_summary

{'invalid_latitude': np.int64(0),
 'invalid_longitude': np.int64(0),
 'missing_datetime': np.int64(0),
 'duplicate_rows': np.int64(0)}

## Cleaning notes

- FIRMS acquisition time is parsed as HHMM and combined with acquisition date into UTC timestamps.
- Coordinates outside valid latitude and longitude ranges are removed.
- Duplicate records are dropped after validation.
- Categorical confidence values such as low, nominal, and high are normalized into a 0-100 score.
- `brightness_primary` uses the best available brightness field across MODIS and VIIRS variants.